# 10 - MiniConvNet checkpoint production (single run, addendum to Phase 2)

**Purpose, and only this purpose**: produce one reusable `.keras` checkpoint for MiniConvNet, because
Phase 1's audit found none exist anywhere in this repo (`models/` is git-ignored, nothing survived
across sessions). Phase 3 (Grad-CAM) and Phase 5 (external generalisation) need a loadable model;
without this notebook they would each trigger their own retrain.

**This is NOT a re-verification of Phase 2's fairness comparison.** `notebooks/09` correctly reuses
MiniConvNet's existing 3-fold CV result for that comparison rather than retraining. This notebook
exists purely to leave a checkpoint file behind for later phases to load.

**Single run, not 3-fold CV.** Exactly one training run, using the identical configuration as the
original successful run: seed 42, `Adam(1e-4, clipnorm=1.0)`, LeakyReLU + he_normal + the widened
64-unit bottleneck, label smoothing 0.05, `ReduceLROnPlateau`, faithful split, 60-epoch budget with
early stopping. Nothing about the architecture or training recipe is changed - the sole purpose of
running it again is to have weights this time.

**Sanity target, not a re-verification target**: the original single `faithful` run scored 0.4063
accuracy (`outputs/experiments_log.csv`, run `miniconvnet_faithful`) - note this is NOT the 0.7059
3-fold CV figure, which pools all 1,000 images across folds. This run should land close to 0.4063,
since it repeats that exact single-run protocol. If it does not land close, **that discrepancy is
reported as-is** in the final cell - it is not silently treated as a bug and re-run until it matches.

**CPU only, nothing executed in this session.** The time-estimate cell prints a number; the decision
to proceed is the runner's, made later.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print()
print('anti-collapse settings (identical to the original successful run):')
for k, v in describe_anti_collapse_settings().items():
    print(f'  {k}: {v}')

In [ ]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets, split_counts
from src.models import build_miniconvnet, count_params, check_param_budget
from src.train_utils import (set_global_seeds, compute_report, compile_model, optimizer_summary,
                             class_weights_for, make_callbacks, make_epoch_timer, save_history,
                             plot_history, final_epoch_summary, run_name_for,
                             estimate_training_time)
from src.evaluate_utils import predict, load_results
from src.finetune_utils import (full_metrics, save_checkpoint_with_metadata,
                                list_local_checkpoints, CHECKPOINTS_LOCAL)

set_global_seeds(SEED)
for k, v in compute_report().items():
    print(f'{k}: {v}')

In [ ]:
SPLIT_FOR_CHECKPOINT = 'faithful'      # identical to the original single run this reproduces

sdf = load_split(SPLIT_FOR_CHECKPOINT)
train_ds, val_ds, test_ds, frames = make_split_datasets(sdf, one_hot=True)   # one-hot: label smoothing
class_weight = class_weights_for(SPLIT_FOR_CHECKPOINT, frames['train']['label'].values)

print(split_counts(sdf).to_string())
print('class_weight:', class_weight if class_weight else 'None (project default for faithful)')

## Time estimate (read this, then decide whether to proceed)

Measured on a real epoch, extrapolated over the 60-epoch budget. Consistent with the original run's
measured cost (~4-6 s/epoch, `outputs/experiments_log.csv` records 4.0 s/epoch / 2.27 min total for
this exact configuration), so this should land in the same few-minute range.

In [ ]:
est = estimate_training_time(
    model_fn=lambda: compile_model(build_miniconvnet(), verbose=False),
    train_ds=train_ds, val_ds=val_ds,
    planned_epochs=EPOCHS_MINICONVNET, n_runs=1)
print()
print('For reference, the original run of this exact config measured 4.0 s/epoch, 2.27 min total.')

## Train (single run) and save the checkpoint

The only step that differs from the original run: the model is saved via
`finetune_utils.save_checkpoint_with_metadata()`, which writes `checkpoints_local/miniconvnet_single_run.keras`
plus a `.json` sidecar recording the config and final metrics - the artefact this whole notebook exists to
produce.

In [ ]:
run_name = 'miniconvnet_single_run'
set_global_seeds(SEED)

model = build_miniconvnet()
budget_check = check_param_budget(model)
compile_model(model)
print('params   :', count_params(model)['total_params'])
print('optimizer:', optimizer_summary(model))

timer = make_epoch_timer(verbose=1)
t0 = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_MINICONVNET,
                    class_weight=class_weight,
                    callbacks=make_callbacks(run_name, timer=timer), verbose=2)
train_seconds = time.time() - t0
save_history(history, run_name, timer=timer)
summary = final_epoch_summary(history, timer=timer)
print()
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
y_true, y_pred, y_prob = predict(model, test_ds)
metrics = full_metrics(y_true, y_pred, y_prob)

print('test metrics:')
for k in ('accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'cohen_kappa', 'mcc',
          'tumour_detection_accuracy', 'subtype_accuracy'):
    print(f'  {k}: {metrics[k]:.4f}')
print()
print('confusion matrix:')
print(np.array(metrics['_confusion_matrix']))

In [ ]:
config = {
    'model': 'MiniConvNet', 'run_type': 'scratch', 'split': SPLIT_FOR_CHECKPOINT,
    'seed': SEED, 'lr': LR_MINICONVNET, 'clipnorm': CLIPNORM,
    'label_smoothing': LABEL_SMOOTHING, 'activation': ACTIVATION,
    'epochs_budget': EPOCHS_MINICONVNET, 'epochs_run': summary['epochs_trained'],
    'class_weight_used': bool(class_weight),
    'purpose': 'checkpoint production for Phase 3/4/5 - NOT the Phase 2 fairness comparison run',
}
ck = save_checkpoint_with_metadata(
    model, run_name, model_name='MiniConvNet', run_type='scratch',
    metrics=metrics, config=config, epochs_run=summary['epochs_trained'],
    extra={'train_seconds': round(train_seconds, 1), 'param_budget_check': budget_check})

tf.keras.backend.clear_session()

## Sanity check against the original single-run number

Compares this run's accuracy against the ORIGINAL single `faithful` run (0.4063,
`outputs/experiments_log.csv` -> `miniconvnet_faithful`) - the correct reference, since both are the
same single-run protocol on the same split. **This is a discrepancy report, not a pass/fail gate**:
if the numbers diverge, that is stated plainly and left for a human to investigate, not silently
retried.

In [ ]:
ORIGINAL_SINGLE_RUN_ACCURACY = 0.4063   # outputs/experiments_log.csv, run 'miniconvnet_faithful'
TOLERANCE = 0.05                        # generous: CPU float non-determinism + no fixed-op determinism

# Cross-check the hardcoded reference against the live file, if present, so this
# cell can never silently compare against a stale number.
try:
    exp_log = pd.read_csv(OUTPUT_ROOT / 'experiments_log.csv')
    row = exp_log[exp_log['model'] == 'miniconvnet_faithful']
    if len(row):
        live_val = float(row.iloc[0]['accuracy'])
        if abs(live_val - ORIGINAL_SINGLE_RUN_ACCURACY) > 1e-3:
            print(f'NOTE: outputs/experiments_log.csv has {live_val:.4f} for miniconvnet_faithful, '
                  f'not the hardcoded {ORIGINAL_SINGLE_RUN_ACCURACY} - using the live value below.')
            ORIGINAL_SINGLE_RUN_ACCURACY = live_val
except FileNotFoundError:
    print('outputs/experiments_log.csv not found on this machine - using the hardcoded reference.')

delta = metrics['accuracy'] - ORIGINAL_SINGLE_RUN_ACCURACY
print(f"this run's accuracy       : {metrics['accuracy']:.4f}")
print(f'original single-run accuracy : {ORIGINAL_SINGLE_RUN_ACCURACY:.4f}')
print(f'delta                        : {delta:+.4f}')
if abs(delta) <= TOLERANCE:
    print(f'CLOSE to the original single-run result (within {TOLERANCE}) - reproduction looks sound.')
else:
    print(f'!!! DIVERGES from the original single-run result by more than {TOLERANCE}.')
    print('    This is reported as a discrepancy, not assumed to be a bug and re-run.')
    print('    Possible causes to investigate: TensorFlow/library version differences producing')
    print('    non-identical results despite the fixed seed, hardware-dependent floating-point')
    print('    non-determinism, or a real difference in the environment vs. the original run.')

## Final checklist

In [ ]:
ck_table = list_local_checkpoints()
print('CHECKPOINT BACKUP CHECKLIST (checkpoints_local/ is git-ignored - back these up manually)')
print('=' * 90)
for _, r in ck_table.iterrows():
    flag = '' if r['metadata_present'] else '  <-- MISSING METADATA SIDECAR'
    print(f"  {r['checkpoint']}  ({r['size_mb']} MB){flag}")
print()
print(f"checkpoint file : {ck['checkpoint']}")
print(f"metadata file   : {ck['metadata']}")
print(f"size            : {ck['size_mb']} MB")
print(f"accuracy        : {metrics['accuracy']:.4f}  (vs original single-run "
      f"{ORIGINAL_SINGLE_RUN_ACCURACY:.4f}, delta {delta:+.4f})")
print('next: back this checkpoint up manually (Drive / external disk) - it will not be')
print('picked up by git. Phase 3 and Phase 5 should load it from checkpoints_local/ rather')
print('than retraining.')